In [1]:
!pip install -q transformers accelerate

In [2]:
import torch

In [3]:
print(torch.__version__)

2.8.0+cu128


In [4]:
print(torch.cuda.device_count())

2


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
model.eval()
print(f"Model loaded on cuda:0")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded on cuda:0
Params: 494.0M


In [8]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
print(f"Input shape: {inputs.input_ids.shape}")

with torch.no_grad():
    out = model(**inputs, use_cache=True)

kv = out.past_key_values
print(f"\nType: {type(kv).__name__}")
print(f"Number of layers: {len(kv)}")

# Helper that works across Transformers versions
def get_kv(cache, layer_idx):
    if hasattr(cache, "layers"):              # newest API
        return cache.layers[layer_idx].keys, cache.layers[layer_idx].values
    elif hasattr(cache, "key_cache"):         # older DynamicCache
        return cache.key_cache[layer_idx], cache.value_cache[layer_idx]
    else:                                      # legacy tuple format
        return cache[layer_idx][0], cache[layer_idx][1]

k0, v0 = get_kv(kv, 0)
print(f"K shape (layer 0): {k0.shape}")
print(f"V shape (layer 0): {v0.shape}")
print(f"dtype: {k0.dtype}")



Input shape: torch.Size([1, 5])

Type: DynamicCache
Number of layers: 24
K shape (layer 0): torch.Size([1, 2, 5, 64])
V shape (layer 0): torch.Size([1, 2, 5, 64])
dtype: torch.float16


In [9]:
total_bytes = 0
for i in range(len(kv)):
    k, v = get_kv(kv, i)
    total_bytes += k.numel() * k.element_size()
    total_bytes += v.numel() * v.element_size()

seq_len = inputs.input_ids.shape[1]
print(f"KV cache size: {total_bytes / 1024 / 1024:.2f} MB for {seq_len} tokens")
print(f"Per token: {total_bytes / seq_len / 1024:.2f} KB")

KV cache size: 0.06 MB for 5 tokens
Per token: 12.00 KB


In [10]:
cfg = model.config
print(f"num_attention_heads: {cfg.num_attention_heads}")
print(f"num_key_value_heads: {cfg.num_key_value_heads}")
print(f"hidden_size: {cfg.hidden_size}")
print(f"num_hidden_layers: {cfg.num_hidden_layers}")
print(f"head_dim: {cfg.hidden_size // cfg.num_attention_heads}")

num_attention_heads: 14
num_key_value_heads: 2
hidden_size: 896
num_hidden_layers: 24
head_dim: 64


In [11]:
import time

def time_prefill(prompt_len, n_warmup=2, n_runs=5):
    # Make a fake prompt of exactly prompt_len tokens
    input_ids = torch.randint(0, 1000, (1, prompt_len), device="cuda:0")
    
    # Warmup
    for _ in range(n_warmup):
        with torch.no_grad():
            _ = model(input_ids, use_cache=True)
    torch.cuda.synchronize()
    
    # Time it
    start = time.perf_counter()
    for _ in range(n_runs):
        with torch.no_grad():
            out = model(input_ids, use_cache=True)
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_runs
    return elapsed, out.past_key_values

print(f"{'prompt_len':>12} {'prefill_ms':>12} {'tokens/sec':>12}")
for L in [128, 512, 1024, 2048, 4096]:
    t, _ = time_prefill(L)
    print(f"{L:>12} {t*1000:>12.2f} {L/t:>12.0f}")

  prompt_len   prefill_ms   tokens/sec
         128        11.82        10833
         512        12.13        42221
        1024        12.28        83388
        2048        12.90       158760
        4096        24.56       166806


In [12]:
def time_decode(prompt_len, n_decode=50, n_warmup=2):
    input_ids = torch.randint(0, 1000, (1, prompt_len), device="cuda:0")
    
    # Prefill once to get the KV cache
    with torch.no_grad():
        out = model(input_ids, use_cache=True)
    past = out.past_key_values
    next_token = out.logits[:, -1:].argmax(dim=-1)
    
    # Warmup decode steps
    for _ in range(n_warmup):
        with torch.no_grad():
            out = model(next_token, past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_token = out.logits[:, -1:].argmax(dim=-1)
    torch.cuda.synchronize()
    
    # Time decode
    start = time.perf_counter()
    for _ in range(n_decode):
        with torch.no_grad():
            out = model(next_token, past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_token = out.logits[:, -1:].argmax(dim=-1)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_decode
    return elapsed

print(f"{'ctx_len':>12} {'decode_ms/tok':>15} {'tokens/sec':>12}")
for L in [128, 512, 1024, 2048, 4096]:
    t = time_decode(L)
    print(f"{L:>12} {t*1000:>15.2f} {1/t:>12.0f}")

     ctx_len   decode_ms/tok   tokens/sec
         128           10.57           95
         512           10.70           93
        1024           10.69           94
        2048           10.58           94
        4096           10.58           94


In [13]:
model_gpu1 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:1",
)
model_gpu1.eval()
print("Model loaded on cuda:1")
print(f"GPU 0 memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU 1 memory: {torch.cuda.memory_allocated(1) / 1e9:.2f} GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded on cuda:1
GPU 0 memory: 1.07 GB
GPU 1 memory: 0.99 GB


In [14]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")

@torch.no_grad()
def generate_single_gpu(input_ids, max_new_tokens=20):
    out = model(input_ids, use_cache=True)
    past = out.past_key_values
    next_token = out.logits[:, -1:].argmax(dim=-1)
    generated = [next_token]
    
    for _ in range(max_new_tokens - 1):
        out = model(next_token, past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_token = out.logits[:, -1:].argmax(dim=-1)
        generated.append(next_token)
    
    return torch.cat(generated, dim=1)

baseline = generate_single_gpu(inputs.input_ids, max_new_tokens=20)
baseline_text = tokenizer.decode(baseline[0])
print(f"Baseline (single GPU): {baseline_text!r}")

Baseline (single GPU): ' Paris. It is the largest city in Europe and the second largest in the world. It is also'


In [15]:
from transformers import DynamicCache

def move_cache_to_device(cache, device):
    """Move a DynamicCache from one GPU to another by moving each layer's K and V tensors."""
    new_cache = DynamicCache()
    for i in range(len(cache)):
        k, v = get_kv(cache, i)
        k_new = k.to(device, non_blocking=True)
        v_new = v.to(device, non_blocking=True)
        new_cache.update(k_new, v_new, i)
    torch.cuda.synchronize()
    return new_cache

In [16]:
@torch.no_grad()
def generate_disaggregated(input_ids_gpu0, max_new_tokens=20):
    # PREFILL on GPU 0
    out = model(input_ids_gpu0, use_cache=True)
    past_gpu0 = out.past_key_values
    
    # TRANSFER KV cache to GPU 1
    past_gpu1 = move_cache_to_device(past_gpu0, "cuda:1")
    
    # First generated token comes from prefill — move logits to GPU 1
    next_token = out.logits[:, -1:].argmax(dim=-1).to("cuda:1")
    generated = [next_token]
    
    # DECODE on GPU 1
    for _ in range(max_new_tokens - 1):
        out = model_gpu1(next_token, past_key_values=past_gpu1, use_cache=True)
        past_gpu1 = out.past_key_values
        next_token = out.logits[:, -1:].argmax(dim=-1)
        generated.append(next_token)
    
    return torch.cat(generated, dim=1)

disagg = generate_disaggregated(inputs.input_ids, max_new_tokens=20)
disagg_text = tokenizer.decode(disagg[0])
print(f"Disaggregated:         {disagg_text!r}")
print(f"\nMatch: {torch.equal(baseline.to('cuda:1'), disagg)}")

Disaggregated:         ' Paris. It is the largest city in Europe and the second largest in the world. It is also'

Match: True


In [17]:
print(f"GPU 0 -> GPU 1 peer access: {torch.cuda.can_device_access_peer(0, 1)}")
print(f"GPU 1 -> GPU 0 peer access: {torch.cuda.can_device_access_peer(1, 0)}")

GPU 0 -> GPU 1 peer access: True
GPU 1 -> GPU 0 peer access: True


In [18]:
import time

def bench_transfer(size_mb):
    n_floats = (size_mb * 1024 * 1024) // 2  # fp16 = 2 bytes
    x = torch.randn(n_floats, dtype=torch.float16, device="cuda:0")
    
    # Warmup
    for _ in range(3):
        y = x.to("cuda:1")
    torch.cuda.synchronize()
    
    # Time it
    n_runs = 20
    start = time.perf_counter()
    for _ in range(n_runs):
        y = x.to("cuda:1", non_blocking=True)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_runs
    
    bytes_moved = n_floats * 2
    bandwidth_gbs = bytes_moved / elapsed / 1e9
    print(f"  {size_mb:>5} MB: {elapsed*1000:>7.3f} ms  -> {bandwidth_gbs:>6.1f} GB/s")
    return bandwidth_gbs

print("Transfer bandwidth GPU 0 -> GPU 1:")
for sz in [1, 10, 100, 500, 1000]:
    bench_transfer(sz)

Transfer bandwidth GPU 0 -> GPU 1:
      1 MB:   0.020 ms  ->   52.3 GB/s
     10 MB:   0.055 ms  ->  189.2 GB/s
    100 MB:   0.411 ms  ->  255.4 GB/s
    500 MB:   1.985 ms  ->  264.2 GB/s
   1000 MB:   3.954 ms  ->  265.2 GB/s
